#**Wide-Field Imager (WFI) Time Series Dataset for Burned Area Mapping in the Brazilian Cerrado**

Available in: https://doi.org/10.57760/sciencedb.14040

Supplement to: https://doi.org/10.1080/20964471.2025.2463731

**Dataset structure**
-----------------
Temporal range:
    January 1, 2020 (200101; yymmdd) – December 31, 2022 (221231; yymmdd)

Bands and spectral indices columns:

    Band / Index    Prefix    Column range
    --------------------------------------
    Blue            a_        [2 : 1096]
    Green           b_        [1097 : 2191]
    Red             c_        [2192 : 3286]
    NIR             d_        [3287 : 4381]
    BAI             e_        [4382 : 5476]
    EVI             f_        [5477 : 6571]
    GEMI            g_        [6572 : 7666]
    NDVI            h_        [7667 : 8761]
    NDWI            i_        [8762 : 9856]

Date of burn scars and labels columns

    Date / Labels    Column index
    --------------------------------------
    date_20          [9856]
    date_21          [9857]
    date_22          [9858]
    20_labels        [9859]
    21_labels        [9860]
    22_labels        [9861]

**Dataset download and conversion to parquet**

In [ ]:
# ============================================
# UTILITY FUNCTIONS
# ============================================

import requests
from tqdm.auto import tqdm

def download(url: str, output: str) -> None:
    """
    Download a file from a URL with a progress bar.

    Parameters:
    -----------
    url : str
        URL of the file to download
    output : str
        Local path where the file will be saved
    """
    response = requests.get(url, stream=True)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))

    with open(output, "wb") as f, tqdm(
        total=total_size,
        unit="B",
        unit_scale=True,
        desc=output.split("/")[-1],  # Show only filename in progress bar
    ) as pbar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            pbar.update(len(chunk))

**Data preparation**

In [ ]:
# ============================================
# DATA PREPARATION FOR CLASSIFICATION
# ============================================

import pandas as pd
import numpy as np
import os

parquet_path = "/path/ScienceDB/Dataset/Dataset_WFI.parquet" # update with your directory

print("Loading data...")
df = pd.read_parquet(parquet_path)
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]:,} columns")

label_cols = ["date_20", "date_21", "date_22", "20_labels", "21_labels", "22_labels"]
df_labels = df[label_cols].copy()

BASE_DIR = "/path/ScienceDB/Dataset" # update with your directory
os.makedirs(BASE_DIR, exist_ok=True)

# ----------------------------------------------------------------------------
# FUNCTIONS FOR EACH INDEX (NIR, NDWI, and EVI)
# ----------------------------------------------------------------------------

def get_nir_cols(year):
    return sorted([col for col in df.columns if col.startswith(f'd_{str(year)[-2:]}')])

def get_ndwi_cols(year):
    return sorted([col for col in df.columns if col.startswith(f'i_{str(year)[-2:]}')])

def get_evi_cols(year):
    return sorted([col for col in df.columns if col.startswith(f'f_{str(year)[-2:]}')])

def aggregate_series(series, cols, period):
    n = len(cols)
    n_groups = (n + period - 1) // period

    means, medians, stds, mins, p25s, amps = [], [], [], [], [], []

    for i in range(n_groups):
        start = i * period
        end = min((i + 1) * period, n)
        group = series[cols[start:end]]

        means.append(group.mean())
        medians.append(group.median())
        stds.append(group.std())
        mins.append(group.min())
        p25s.append(group.quantile(0.25))
        amps.append(group.max() - group.min())

    return {'mean': np.array(means), 'median': np.array(medians),
            'std': np.array(stds), 'min': np.array(mins),
            'p25': np.array(p25s), 'amp': np.array(amps)}

def create_aggregated_dataset_by_index(year, period, get_cols_func, index_name):
    cols = get_cols_func(year)
    if not cols:
        return None

    data = df[cols].values
    results = []

    for i in range(len(data)):
        series = pd.Series(data[i], index=cols)
        metrics = aggregate_series(series, cols, period)

        row = {}
        for key, values in metrics.items():
            for k, val in enumerate(values):
                row[f"{index_name}_{key}_{k+1:03d}"] = val
        results.append(row)

    return pd.DataFrame(results)

def create_complete_aggregated_dataset(year, period):
    df_year = None

    # NIR
    df_nir = create_aggregated_dataset_by_index(year, period, get_nir_cols, 'NIR')
    if df_nir is not None:
        df_year = df_nir

    # NDWI
    df_ndwi = create_aggregated_dataset_by_index(year, period, get_ndwi_cols, 'NDWI')
    if df_ndwi is not None:
        if df_year is None:
            df_year = df_ndwi
        else:
            df_year = pd.concat([df_year, df_ndwi], axis=1)

    # EVI
    df_evi = create_aggregated_dataset_by_index(year, period, get_evi_cols, 'EVI')
    if df_evi is not None:
        if df_year is None:
            df_year = df_evi
        else:
            df_year = pd.concat([df_year, df_evi], axis=1)

    if df_year is None:
        return None

    year_str = str(year)[-2:]
    df_year['date'] = df_labels[f'date_{year_str}']
    df_year['label'] = df_labels[f'{year_str}_labels']
    df_year['year'] = year

    return df_year

# ----------------------------------------------------------------------------
# CHECK IF DATASETS ALREADY EXIST
# ----------------------------------------------------------------------------

daily_path = os.path.join(BASE_DIR, "daily_nir_ndwi_evi.parquet")
sixteen_path = os.path.join(BASE_DIR, "sixteen_nir_ndwi_evi.parquet")
thirty_path = os.path.join(BASE_DIR, "thirty_nir_ndwi_evi.parquet")

daily_exists = os.path.exists(daily_path)
sixteen_exists = os.path.exists(sixteen_path)
thirty_exists = os.path.exists(thirty_path)

if daily_exists and sixteen_exists and thirty_exists:
    print("\n" + "=" * 60)
    print("ALL DATASETS ALREADY EXIST")
    print("=" * 60)
    print(f"\n  {daily_path}")
    print(f"  {sixteen_path}")
    print(f"  {thirty_path}")
    print("\nSkipping dataset creation. Files already present.")
    print("=" * 60)
    exit(0)

# ----------------------------------------------------------------------------
# DAILY DATASET (NIR + NDWI + EVI)
# ----------------------------------------------------------------------------

if not daily_exists:
    print("\nCreating daily_nir_ndwi_evi.parquet...")
    df_daily_list = []

    for year in [2020, 2021, 2022]:
        year_str = str(year)[-2:]
        df_year = None

        # NIR
        nir_cols = get_nir_cols(year)
        if nir_cols:
            df_year = df[nir_cols].copy()

        # NDWI
        ndwi_cols = get_ndwi_cols(year)
        if ndwi_cols:
            df_temp = df[ndwi_cols].copy()
            if df_year is None:
                df_year = df_temp
            else:
                df_year = pd.concat([df_year, df_temp], axis=1)

        # EVI
        evi_cols = get_evi_cols(year)
        if evi_cols:
            df_temp = df[evi_cols].copy()
            if df_year is None:
                df_year = df_temp
            else:
                df_year = pd.concat([df_year, df_temp], axis=1)

        if df_year is not None:
            df_year['date'] = df_labels[f'date_{year_str}']
            df_year['label'] = df_labels[f'{year_str}_labels']
            df_year['year'] = year
            df_daily_list.append(df_year)
            print(f"  Year {year}: {df_year.shape[0]:,} rows, {df_year.shape[1]:,} columns")

    if df_daily_list:
        df_daily = pd.concat(df_daily_list, ignore_index=True)
        df_daily.to_parquet(daily_path)
        print(f"daily_nir_ndwi_evi.parquet: {df_daily.shape[0]:,} x {df_daily.shape[1]:,}")
else:
    print(f"\ndaily_nir_ndwi_evi.parquet already exists. Skipping...")

# ----------------------------------------------------------------------------
# 16-DAY DATASET
# ----------------------------------------------------------------------------

if not sixteen_exists:
    print("\nCreating sixteen_nir_ndwi_evi.parquet...")
    df_sixteen = None

    for year in [2020, 2021, 2022]:
        print(f"  Processing {year}...")
        df_year = create_complete_aggregated_dataset(year, 16)
        if df_year is not None:
            df_sixteen = df_year if df_sixteen is None else pd.concat([df_sixteen, df_year], ignore_index=True)

    if df_sixteen is not None:
        df_sixteen.to_parquet(sixteen_path)
        print(f"sixteen_nir_ndwi_evi.parquet: {df_sixteen.shape[0]:,} x {df_sixteen.shape[1]:,}")
    else:
        print("Error: No 16-day data generated.")
else:
    print("\nsixteen_nir_ndwi_evi.parquet already exists. Skipping...")

# ----------------------------------------------------------------------------
# 30-DAY DATASET
# ----------------------------------------------------------------------------

if not thirty_exists:
    print("\nCreating thirty_nir_ndwi_evi.parquet...")
    df_thirty = None

    for year in [2020, 2021, 2022]:
        print(f"  Processing {year}...")
        df_year = create_complete_aggregated_dataset(year, 30)
        if df_year is not None:
            df_thirty = df_year if df_thirty is None else pd.concat([df_thirty, df_year], ignore_index=True)

    if df_thirty is not None:
        df_thirty.to_parquet(thirty_path)
        print(f"thirty_nir_ndwi_evi.parquet: {df_thirty.shape[0]:,} x {df_thirty.shape[1]:,}")
    else:
        print("Error: No 30-day data generated.")
else:
    print("\nthirty_nir_ndwi_evi.parquet already exists. Skipping...")

# ----------------------------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

if daily_exists:
    df_daily = pd.read_parquet(daily_path)
    print(f"\n✓ daily_nir_ndwi_evi.parquet: {df_daily.shape[0]:,} x {df_daily.shape[1]:,} (already existed)")
elif df_daily_list:
    print(f"\n✓ daily_nir_ndwi_evi.parquet: {df_daily.shape[0]:,} x {df_daily.shape[1]:,} (created)")

if sixteen_exists:
    df_sixteen = pd.read_parquet(sixteen_path)
    print(f"✓ sixteen_nir_ndwi_evi.parquet: {df_sixteen.shape[0]:,} x {df_sixteen.shape[1]:,} (already existed)")
elif df_sixteen is not None:
    print(f"✓ sixteen_nir_ndwi_evi.parquet: {df_sixteen.shape[0]:,} x {df_sixteen.shape[1]:,} (created)")

if thirty_exists:
    df_thirty = pd.read_parquet(thirty_path)
    print(f"✓ thirty_nir_ndwi_evi.parquet: {df_thirty.shape[0]:,} x {df_thirty.shape[1]:,} (already existed)")
elif df_thirty is not None:
    print(f"✓ thirty_nir_ndwi_evi.parquet: {df_thirty.shape[0]:,} x {df_thirty.shape[1]:,} (created)")

print(f"\nDirectory: {BASE_DIR}")
print("=" * 60)

Loading data...
Dataset loaded: 38,957 rows, 9,862 columns

ALL DATASETS ALREADY EXIST

  /home/jovyan/ScienceDB/Dataset/daily_nir_ndwi_evi.parquet
  /home/jovyan/ScienceDB/Dataset/sixteen_nir_ndwi_evi.parquet
  /home/jovyan/ScienceDB/Dataset/thirty_nir_ndwi_evi.parquet

Skipping dataset creation. Files already present.

daily_nir_ndwi_evi.parquet already exists. Skipping...

sixteen_nir_ndwi_evi.parquet already exists. Skipping...

thirty_nir_ndwi_evi.parquet already exists. Skipping...

DATASET SUMMARY

✓ daily_nir_ndwi_evi.parquet: 116,871 x 1,828 (already existed)
✓ sixteen_nir_ndwi_evi.parquet: 116,871 x 417 (already existed)
✓ thirty_nir_ndwi_evi.parquet: 116,871 x 237 (already existed)

Directory: /home/jovyan/ScienceDB/Dataset


**Complete pipeline with Random Forest**

In [ ]:
# ============================================
# RANDOM FOREST TRAINING - WITH OPTUNA OPTIMIZATION
# ============================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix
)
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import joblib
import os
import time
import gc
import sys
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURATION
# ============================================

BASE_DIR = "path/ScienceDB/Dataset"
MODEL_DIR = "path/ScienceDB/RF"
os.makedirs(MODEL_DIR, exist_ok=True)

# Optuna parameters
N_TRIALS = {
    'daily': 50,
    'sixteen': 40,
    'thirty': 30,
}

# Number of top features to select for final training
TOP_FEATURES = {
    'daily': 100,
    'sixteen': 25,
    'thirty': 25,
}

DATASETS = {
    'daily': {
        'path': os.path.join(BASE_DIR, 'daily_nir_ndwi_evi.parquet'),
        'label': 'Daily',
        'top_n': 100,
        'n_trials': 20,
    },
    'sixteen': {
        'path': os.path.join(BASE_DIR, 'sixteen_nir_ndwi_evi.parquet'),
        'label': '16-Day',
        'top_n': 25,
        'n_trials': 20,
    },
    'thirty': {
        'path': os.path.join(BASE_DIR, 'thirty_nir_ndwi_evi.parquet'),
        'label': '30-Day',
        'top_n': 25,
        'n_trials': 20,
    }
}

# ============================================
# LOGGING UTILITY
# ============================================

def log_progress(message, level="INFO"):
    timestamp = time.strftime("%H:%M:%S")
    print(f"[{timestamp}] [{level}] {message}")
    sys.stdout.flush()

def log_memory(step=""):
    try:
        import psutil
        process = psutil.Process(os.getpid())
        mem = process.memory_info().rss / 1024 / 1024 / 1024
        log_progress(f"Memory: {mem:.2f} GB ({step})", "MEMORY")
        return mem
    except ImportError:
        return None

def log_time(start_time, step=""):
    elapsed = time.time() - start_time
    log_progress(f"Time: {elapsed:.2f}s ({step})", "TIME")
    return elapsed

def log_section(title):
    print("\n" + "=" * 80)
    print(f"  {title}")
    print("=" * 80)
    sys.stdout.flush()

def log_subsection(title):
    print("\n" + "-" * 60)
    print(f"  {title}")
    print("-" * 60)
    sys.stdout.flush()

# ============================================
# UTILITY FUNCTIONS
# ============================================

def load_data(dataset_path):
    log_progress(f"Loading data from {os.path.basename(dataset_path)}...")

    df = pd.read_parquet(dataset_path)
    log_progress(f"  Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

    df_2020 = df[df['year'] == 2020].copy()
    df_2021 = df[df['year'] == 2021].copy()
    df_2022 = df[df['year'] == 2022].copy()

    feature_cols = [col for col in df.columns if col not in ['date', 'label', 'year']]
    log_progress(f"  Features: {len(feature_cols):,}")

    df_train = pd.concat([df_2020, df_2021], ignore_index=True)
    df_train['label_binary'] = df_train['label'].apply(lambda x: 1 if x == 'tb' else 0)

    df_2022['label_binary'] = df_2022['label'].apply(lambda x: 1 if x == 'tb' else 0)

    log_progress(f"  Training samples: {len(df_train):,}")
    log_progress(f"  Test samples: {len(df_2022):,}")

    tb_train = len(df_train[df_train['label'] == 'tb'])
    pb_train = len(df_train[df_train['label'] == 'pb'])
    nb_train = len(df_train[df_train['label'] == 'nb'])
    log_progress(f"  Train classes - TB: {tb_train:,}, PB: {pb_train:,}, NB: {nb_train:,}")

    return df_train, df_2022, feature_cols

def objective(trial, X_train, y_train, X_val, y_val):
    """Optuna objective function."""

    # Suggest hyperparameters
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400, step=50),
        'max_depth': trial.suggest_int('max_depth', 4, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
    }

    # Train model
    model = RandomForestClassifier(**params, random_state=42, n_jobs=4)
    model.fit(X_train, y_train)

    # Predict on validation set
    y_pred = model.predict(X_val)

    # Return F1 score (maximize)
    return f1_score(y_val, y_pred)

def get_feature_importance_with_optuna(df_train, feature_cols, dataset_name, n_trials=50):
    """
    Perform Optuna optimization to find best parameters and get feature importance.
    Uses permutation importance for robust feature selection.
    """
    log_subsection("PHASE 1: OPTUNA OPTIMIZATION + VALIDATION")

    log_progress(f"Running Optuna with {n_trials} trials...")

    log_progress("Preparing feature matrix...")
    start_time = time.time()

    X = df_train[feature_cols].fillna(0)
    y = df_train['label']

    log_progress(f"  X shape: {X.shape[0]:,} x {X.shape[1]:,}")
    log_progress(f"  y shape: {y.shape[0]:,}")

    # Convert to binary (TB=1, PB+NB=0)
    y_binary = np.array([1 if l == 'tb' else 0 for l in y])

    # Single split for validation
    X_train, X_val, y_train, y_val = train_test_split(
        X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
    )

    log_progress(f"  Training set: {len(X_train):,} samples")
    log_progress(f"  Validation set: {len(X_val):,} samples")
    log_time(start_time, "Data preparation")

    log_progress("Running Optuna optimization...")
    log_memory("Before Optuna")
    optuna_start = time.time()

    # Create Optuna study with pruning
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(
            n_startup_trials=5,
            n_warmup_steps=10,
            interval_steps=1
        )
    )

    # Run optimization
    study.optimize(
        lambda trial: objective(trial, X_train, y_train, X_val, y_val),
        n_trials=n_trials,
        show_progress_bar=True
    )

    log_time(optuna_start, "Optuna completed")
    log_memory("After Optuna")

    # Get best parameters
    best_params = study.best_params
    best_score = study.best_value

    log_progress(f"  Best Validation F1-Score: {best_score:.4f}")
    log_progress(f"  Best parameters: {best_params}")

    # Train model with best parameters
    log_progress("Training model with best parameters...")
    best_model = RandomForestClassifier(**best_params, random_state=42, n_jobs=4)
    best_model.fit(X_train, y_train)

    # Validate on validation set
    y_pred = best_model.predict(X_val)

    # IMPORTANT: Save copy before deleting
    val_true = y_val.copy()
    val_pred = y_pred.copy()

    val_acc = accuracy_score(y_val, y_pred)
    val_f1 = f1_score(y_val, y_pred)
    val_prec = precision_score(y_val, y_pred)
    val_rec = recall_score(y_val, y_pred)

    log_progress(f"  Validation Results:")
    log_progress(f"    Accuracy: {val_acc:.4f}")
    log_progress(f"    Precision: {val_prec:.4f}")
    log_progress(f"    Recall: {val_rec:.4f}")
    log_progress(f"    F1-Score: {val_f1:.4f}")

    # Get feature importance using permutation importance (robust)
    log_progress("Calculating permutation importance...")
    perm_start = time.time()

    perm_importance = permutation_importance(
        best_model, X_train, y_train,
        n_repeats=10,
        random_state=42,
        n_jobs=4,
        scoring='f1'
    )

    log_time(perm_start, "Permutation importance completed")

    # Create importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': perm_importance.importances_mean,
        'std': perm_importance.importances_std,
    })
    importance_df = importance_df.sort_values('importance', ascending=False)
    importance_df['importance_pct'] = importance_df['importance'] / importance_df['importance'].sum() * 100

    # Clean up memory AFTER saving all needed values
    log_progress("Cleaning up memory...")
    del X_train, X_val, y_train, y_val
    gc.collect()
    log_memory("After cleanup")

    return {
        'best_params': best_params,
        'best_score': best_score,
        'study': study,
        'model': best_model,
        'importance_df': importance_df,
        'val_true': val_true,
        'val_pred': val_pred,
        'feature_cols': feature_cols,
    }

def train_final_with_top_features(df_train, df_test, feature_cols, top_features, best_params, dataset_name):
    """Train final model using only top features with best parameters and generalize to 2022."""
    log_subsection(f"PHASE 2: FINAL TRAINING WITH TOP {len(top_features)} FEATURES")

    log_progress(f"Using {len(top_features)} top features for final training")
    log_progress(f"Best parameters: {best_params}")
    log_progress(f"Top 5 features: {top_features[:5]}")

    log_progress("Preparing full training data with top features...")
    start_time = time.time()

    X_full = df_train[top_features].fillna(0)
    y_full = df_train['label']

    y_full_binary = np.array([1 if l == 'tb' else 0 for l in y_full])

    log_progress(f"  Full training set: {X_full.shape[0]:,} samples x {X_full.shape[1]:,} features")
    log_progress("Training final model with top features...")
    log_memory("Before final training")

    model = RandomForestClassifier(**best_params, random_state=42, n_jobs=4)
    model.fit(X_full, y_full_binary)

    log_time(start_time, "Final model training")
    log_memory("After final training")

    log_progress("Generalizing to 2022...")
    X_test = df_test[top_features].fillna(0)
    log_progress(f"  Test set: {X_test.shape[0]:,} samples")

    y_pred = model.predict(X_test)
    y_test_binary = df_test['label_binary'].values

    test_acc = accuracy_score(y_test_binary, y_pred)
    test_f1 = f1_score(y_test_binary, y_pred)
    test_prec = precision_score(y_test_binary, y_pred)
    test_rec = recall_score(y_test_binary, y_pred)

    log_progress(f"  Generalization Results (2022):")
    log_progress(f"    Accuracy: {test_acc:.4f}")
    log_progress(f"    Precision: {test_prec:.4f}")
    log_progress(f"    Recall: {test_rec:.4f}")
    log_progress(f"    F1-Score: {test_f1:.4f}")

    log_progress("Cleaning up memory...")
    del X_full, y_full, X_test
    gc.collect()
    log_memory("After cleanup")

    return {
        'model': model,
        'test_true': y_test_binary,
        'test_pred': y_pred,
        'top_features': top_features,
    }

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1_score': f1_score(y_true, y_pred),
        'iou': iou,
        'confusion_matrix': cm,
    }

def print_confusion_matrix(cm, title):
    print(f"\n{title}")
    print("-" * 40)
    print(f"{'':>10} | {'Predicted':>20}")
    print(f"{'':>10} | {'PB+NB':>10} | {'TB':>10}")
    print("-" * 40)
    print(f"{'Actual PB+NB':>10} | {cm[0,0]:>10} | {cm[0,1]:>10}")
    print(f"{'Actual TB':>10} | {cm[1,0]:>10} | {cm[1,1]:>10}")
    print("-" * 40)

def print_feature_ranking(importance_df, top_n=20, title="TOP FEATURES"):
    print(f"\n{'='*80}")
    print(f"{title} - TOP {top_n}")
    print(f"{'='*80}")
    print(f"{'Rank':<6} | {'Feature':<45} | {'Importance':>12} | {'%':>8}")
    print("-" * 80)

    top_features = importance_df.head(top_n)
    for idx, row in top_features.iterrows():
        rank = top_features.index.get_loc(idx) + 1
        feature_name = row['feature'][:45]
        print(f"{rank:<6} | {feature_name:<45} | {row['importance']:>12.6f} | {row['importance_pct']:>7.2f}%")

    print("=" * 80)
    total_pct = top_features['importance'].sum() * 100
    print(f"Total contribution of top {top_n}: {total_pct:.2f}%")
    print(f"Total features: {len(importance_df):,}")
    print("=" * 80)

# ============================================
# MAIN EXECUTION
# ============================================

log_section("RANDOM FOREST WITH OPTUNA OPTIMIZATION")
log_progress(f"Script started at {time.strftime('%Y-%m-%d %H:%M:%S')}")

print("\nOptuna Configuration:")
print("  - Sampler: TPE (Tree-structured Parzen Estimator)")
print("  - Pruner: MedianPruner (early stopping)")
print("  - Objective: Maximize F1-Score")

print(f"\nTrials per dataset:")
for key, value in N_TRIALS.items():
    print(f"  {key}: {value} trials")

print(f"\nTop features selection:")
for key, value in TOP_FEATURES.items():
    print(f"  {key}: {value} features")

print("\nFeature importance method: PERMUTATION IMPORTANCE (robust)")
print("\nNOTE: Using single train/validation split (80/20) - NO cross-validation")

script_start = time.time()
val_results = {}
gen_results = {}

for key, cfg in DATASETS.items():
    log_section(f"PROCESSING: {cfg['label']}")
    dataset_start = time.time()

    # Load data
    df_train, df_test, feature_cols = load_data(cfg['path'])

    # Phase 1: Optuna + Feature Importance
    partial = get_feature_importance_with_optuna(
        df_train, feature_cols, cfg['label'],
        n_trials=cfg['n_trials']
    )

    # Save validation results
    val_results[key] = {
        'name': cfg['label'],
        'val_true': partial['val_true'],
        'val_pred': partial['val_pred'],
        'n_features': len(feature_cols),
        'best_params': partial['best_params'],
        'best_score': partial['best_score'],
        'n_trials': cfg['n_trials'],
    }

    # Get top N features
    top_n = cfg['top_n']
    importance_df = partial['importance_df']
    top_features = importance_df.head(top_n)['feature'].tolist()

    # Print feature ranking
    print_feature_ranking(importance_df, top_n=min(20, top_n), title=f"{cfg['label'].upper()} FEATURES")
    log_progress(f"Selected top {len(top_features)} features for final training")

    # Save feature importance
    importance_df.to_csv(os.path.join(MODEL_DIR, f'feature_importance_{key}_optuna_permutation.csv'), index=False)
    importance_df.head(top_n).to_csv(os.path.join(MODEL_DIR, f'feature_importance_{key}_optuna_top{top_n}_permutation.csv'), index=False)

    # Phase 2: Train final model with top features only
    final = train_final_with_top_features(
        df_train, df_test, feature_cols, top_features,
        partial['best_params'], cfg['label']
    )

    gen_results[key] = {
        'name': cfg['label'],
        'test_true': final['test_true'],
        'test_pred': final['test_pred'],
        'n_features': len(top_features),
        'top_features': top_features,
        'best_params': partial['best_params'],
    }

    # Save model
    joblib.dump(final['model'], os.path.join(MODEL_DIR, f'model_{key}_optuna_top{top_n}_permutation.pkl'))
    joblib.dump(top_features, os.path.join(MODEL_DIR, f'top_features_{key}_optuna_{top_n}_permutation.pkl'))
    joblib.dump(partial['best_params'], os.path.join(MODEL_DIR, f'best_params_{key}_optuna_permutation.pkl'))
    log_progress(f"Model saved: model_{key}_optuna_top{top_n}_permutation.pkl")

    elapsed = time.time() - dataset_start
    log_progress(f"Dataset completed in {elapsed/60:.1f} minutes")

    gc.collect()

total_time = time.time() - script_start
log_section(f"ALL DATASETS COMPLETED IN {total_time/60:.1f} MINUTES")

# ============================================
# OPTUNA RESULTS
# ============================================

log_section("OPTUNA RESULTS - BEST PARAMETERS")
print(f"{'Method':<12} | {'Trials':>10} | {'Best F1':>15} | {'Best Parameters':>40}")
print("-" * 80)

for key, r in val_results.items():
    params_str = str(r['best_params'])
    print(f"{r['name']:<12} | {r['n_trials']:>10} | {r['best_score']:>15.4f} | {params_str:<40}")

# ============================================
# VALIDATION RESULTS
# ============================================

log_section("VALIDATION RESULTS (TB vs PB+NB)")
print(f"{'Method':<12} | {'Features':>10} | {'Accuracy':>10} | {'Precision':>10} | {'Recall':>10} | {'F1-Score':>10} | {'IoU':>10}")
print("-" * 100)

for key, r in val_results.items():
    m = compute_metrics(r['val_true'], r['val_pred'])
    print(
        f"{r['name']:<12} | {r['n_features']:>10} | "
        f"{m['accuracy']:>10.4f} | {m['precision']:>10.4f} | "
        f"{m['recall']:>10.4f} | {m['f1_score']:>10.4f} | {m['iou']:>10.4f}"
    )

print("\n" + "=" * 100)
print("VALIDATION CONFUSION MATRICES")
print("=" * 100)

for key, r in val_results.items():
    m = compute_metrics(r['val_true'], r['val_pred'])
    print_confusion_matrix(m['confusion_matrix'], r['name'])

# ============================================
# GENERALIZATION RESULTS (WITH TOP FEATURES)
# ============================================

log_section("GENERALIZATION RESULTS 2022 - WITH TOP FEATURES (OPTUNA + PERMUTATION)")
print(f"{'Method':<12} | {'Features':>10} | {'Accuracy':>10} | {'Precision':>10} | {'Recall':>10} | {'F1-Score':>10} | {'IoU':>10}")
print("-" * 100)

for key, r in gen_results.items():
    m = compute_metrics(r['test_true'], r['test_pred'])
    print(
        f"{r['name']:<12} | {r['n_features']:>10} | "
        f"{m['accuracy']:>10.4f} | {m['precision']:>10.4f} | "
        f"{m['recall']:>10.4f} | {m['f1_score']:>10.4f} | {m['iou']:>10.4f}"
    )

print("\n" + "=" * 100)
print("GENERALIZATION CONFUSION MATRICES - WITH TOP FEATURES (OPTUNA + PERMUTATION)")
print("=" * 100)

for key, r in gen_results.items():
    m = compute_metrics(r['test_true'], r['test_pred'])
    print_confusion_matrix(m['confusion_matrix'], r['name'])

# ============================================
# TOP FEATURES SUMMARY
# ============================================

log_section("TOP FEATURES USED IN FINAL MODELS (OPTUNA + PERMUTATION)")

for key, cfg in DATASETS.items():
    print(f"\n{cfg['label'].upper()}:")
    print("-" * 60)
    print(f"  Total features available: {val_results[key]['n_features']:,}")
    print(f"  Top features selected: {cfg['top_n']}")
    print(f"  Best parameters: {val_results[key]['best_params']}")
    print(f"  Optuna trials: {cfg['n_trials']}")

    # Load top features
    top_features = joblib.load(os.path.join(MODEL_DIR, f'top_features_{key}_optuna_{cfg["top_n"]}_permutation.pkl'))
    print(f"  Top 10 features:")
    for i, feat in enumerate(top_features[:10], 1):
        print(f"    {i:2d}. {feat[:45]}")

    # Show reduction
    reduction = (1 - cfg['top_n'] / val_results[key]['n_features']) * 100
    print(f"  Feature reduction: {reduction:.1f}%")
    print("-" * 60)

# ============================================
# SUMMARY
# ============================================

log_section("SUMMARY: OPTUNA OPTIMIZATION")

print("\nOptuna Configuration:")
print(f"  - Total trials: {sum(N_TRIALS.values())}")
print(f"  - Sampler: TPE (Tree-structured Parzen Estimator)")
print(f"  - Pruner: MedianPruner (early stopping)")

print("\nTrials per dataset:")
for key, cfg in DATASETS.items():
    print(f"  {cfg['label']}: {cfg['n_trials']} trials")

print("\nBest Parameters Found:")
for key, r in val_results.items():
    print(f"  {r['name']}: {r['best_params']} (F1: {r['best_score']:.4f})")

print("\nFiles saved:")
print("  - Feature importance: feature_importance_*_optuna_permutation.csv")
print("  - Top features: top_features_*_optuna_*_permutation.pkl")
print("  - Best parameters: best_params_*_optuna_permutation.pkl")
print("  - Models: model_*_optuna_top*_permutation.pkl")

print(f"\nTotal time: {total_time/60:.1f} minutes")

# ============================================
# SCRIPT COMPLETED
# ============================================

log_section("SCRIPT COMPLETED")
log_progress(f"Finished at {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 100)


  RANDOM FOREST WITH OPTUNA OPTIMIZATION
[14:20:10] [INFO] Script started at 2026-07-28 14:20:10

Optuna Configuration:
  - Sampler: TPE (Tree-structured Parzen Estimator)
  - Pruner: MedianPruner (early stopping)
  - Objective: Maximize F1-Score

Trials per dataset:
  daily: 50 trials
  sixteen: 40 trials
  thirty: 30 trials

Top features selection:
  daily: 100 features
  sixteen: 25 features
  thirty: 25 features

Feature importance method: PERMUTATION IMPORTANCE (robust)

NOTE: Using single train/validation split (80/20) - NO cross-validation

  PROCESSING: Daily
[14:20:10] [INFO] Loading data from daily_nir_ndwi_evi.parquet...
[14:20:10] [INFO]   Shape: 116,871 rows x 1,828 columns
[14:20:12] [INFO]   Features: 1,825
[14:20:13] [INFO]   Training samples: 77,914
[14:20:13] [INFO]   Test samples: 38,957
[14:20:13] [INFO]   Train classes - TB: 4,618, PB: 4,935, NB: 68,361

------------------------------------------------------------
  PHASE 1: OPTUNA OPTIMIZATION + VALIDATION
------

[I 2026-07-28 14:20:17,064] A new study created in memory with name: no-name-8e06f216-4e0e-486c-8df4-873bb6fbad2d


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-07-28 14:20:54,384] Trial 0 finished with value: 0.7972109238814643 and parameters: {'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7972109238814643.
[I 2026-07-28 14:21:58,984] Trial 1 finished with value: 0.7805456702253856 and parameters: {'n_estimators': 350, 'max_depth': 11, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7972109238814643.
[I 2026-07-28 14:28:32,040] Trial 2 finished with value: 0.736068585425597 and parameters: {'n_estimators': 100, 'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 0 with value: 0.7972109238814643.
[I 2026-07-28 14:28:44,313] Trial 3 finished with value: 0.7276142767689417 and parameters: {'n_estimators': 100, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7972109238814643.
[

[I 2026-07-28 21:28:10,543] A new study created in memory with name: no-name-8d59c8fc-cc1f-4800-a365-436e85ae22a7


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-07-28 21:28:40,761] Trial 0 finished with value: 0.7824019024970273 and parameters: {'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7824019024970273.
[I 2026-07-28 21:29:33,423] Trial 1 finished with value: 0.7570552147239263 and parameters: {'n_estimators': 350, 'max_depth': 11, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7824019024970273.
[I 2026-07-28 21:32:18,289] Trial 2 finished with value: 0.7107750472589792 and parameters: {'n_estimators': 100, 'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 0 with value: 0.7824019024970273.
[I 2026-07-28 21:32:28,114] Trial 3 finished with value: 0.7076326002587322 and parameters: {'n_estimators': 100, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7824019024970273.


[I 2026-07-28 22:58:42,080] A new study created in memory with name: no-name-497dd3b4-1f93-4466-9ce8-e4d7faf61c96


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-07-28 22:59:06,769] Trial 0 finished with value: 0.7723132969034608 and parameters: {'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7723132969034608.
[I 2026-07-28 22:59:49,693] Trial 1 finished with value: 0.7458048477315102 and parameters: {'n_estimators': 350, 'max_depth': 11, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7723132969034608.
[I 2026-07-28 23:01:29,302] Trial 2 finished with value: 0.6851971557853911 and parameters: {'n_estimators': 100, 'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 0 with value: 0.7723132969034608.
[I 2026-07-28 23:01:37,225] Trial 3 finished with value: 0.6733601070950469 and parameters: {'n_estimators': 100, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7723132969034608.
